In [ ]:
!pip install sentence_transformers
!pip install langchain_text_splitters
!pip install transformers torch accelerate bitsandbytes

In [ ]:
!git clone https://github.com/chipslays/russian-recipes-parser.git

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter

import torch
import json
import glob
import os

In [ ]:
import json
import glob
import os


# Путь к папке с рецептами (после клонирования репозитория)
recipes_path = "russian-recipes-parser/storage/recipes/[1-3][0-9][0-9]/*.json"

parsed_recipe_list = []

for filepath in glob.glob(recipes_path):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            recipe_data = json.load(f)
        parsed_recipe = {
            "title": recipe_data.get('title', ''),
            "ingredients": recipe_data.get('ingredients', ''),
            "instruction": recipe_data.get('instruction', ''),
            "full_text": "",
        }

        # Собираем полный текст рецепта из полей JSON
        full_recipe_text = f"{recipe_data.get('title', '')}\n\n"
  
        ingredient_list = recipe_data.get('ingredients', '')[0]
        full_recipe_text += f"{ingredient_list['name']}:\n"
        for index, ingredient in enumerate(ingredient_list['list']):
          full_recipe_text += f"{index + 1}) {ingredient['name']} {ingredient['value'] or ''} {ingredient['type'] or ''}\n"
      
        full_recipe_text += f"\nПриготовление:\n" # Здесь лежат шаги
        for index, step in enumerate(recipe_data.get('instruction', '')):
          full_recipe_text += f"{index + 1}) {step['text']}\n"
        parsed_recipe["full_text"] = full_recipe_text
        parsed_recipe_list.append(parsed_recipe)

    except Exception as e:
        print(f"Ошибка при обработке файла {filepath}: {e}")

print(f"Загружено {len(parsed_recipe_list)} рецептов.")
# Посмотрим пример
print("\n--- Пример рецепта ---")
print(parsed_recipe_list[0]["full_text"])

In [ ]:
class RecipeChunker:
    """Создает чанки разного масштаба для одного рецепта"""
    
    def __init__(self, model_name="deepvk/USER2-small", truncate_dim=128):
        self.model = SentenceTransformer(model_name, truncate_dim=truncate_dim)
    
    def chunk_recipe(self, parsed_recipe, max_chunk_size=612):
        chunks = []
        
        # Уровень 1: Целый рецепт (глобальный контекст)
        chunks.append({
            'text': parsed_recipe.get('full_text', ''),
            'scale': 'full',
            'recipe_name': parsed_recipe.get('name', '')
        })
        
        # Уровень 2: Секции (ингредиенты, шаги)
        if parsed_recipe['ingredients']:
          ingredient_list = parsed_recipe['ingredients'][0]
          ing_text = f"Ингредиенты для рецепта \"{parsed_recipe.get('name', '')}\":\n"
          for index, ingredient in enumerate(ingredient_list['list']):
            ing_text += f"{index + 1}) {ingredient['name']} {ingredient['value'] or ''} {ingredient['type'] or ''}\n"
          chunks.append({
            'text': ing_text,
            'scale': 'section',
            'type': 'ingredients',
            'recipe_name': parsed_recipe.get('name', '')
          })
        
        # Уровень 3: Отдельные шаги (детали)
        # Не нравится везде префикс
        if parsed_recipe['instruction']:
          splitter = RecursiveCharacterTextSplitter(
              chunk_size=max_chunk_size,
              chunk_overlap=30,
              separators=[". ", ", ", " "]
          )
          for index, step in enumerate(parsed_recipe['instruction']):
            step_text = f"Инструкции для рецепта \"{parsed_recipe.get('name', '')}\":\n"
            step_text += f"{index + 1}) {step['text']}\n"
            if len(step_text) <= max_chunk_size:
              chunks.append({
                'text': step_text,
                'scale': 'step',
                'step_num': index + 1,
                'recipe_name': parsed_recipe.get('name', '')
              })
              continue
            sub_chunks = splitter.split_text(step_text)
            for part, sub in enumerate(sub_chunks):
              chunks.append({
                'text': sub,
                'type': 'step_part',
                'step': index + 1,
                'part': part + 1,
                'recipe': parsed_recipe.get('name', '')
              })

        return chunks

In [ ]:
# Использование
chunker = RecipeChunker()
chunks = []
for parsed_recipe in parsed_recipe_list:
  chunks.extend(chunker.chunk_recipe(parsed_recipe))

print(f"Создано {len(chunks)} чанков разного масштаба")
for i in range(3):
  chunk = chunks[i]
  print(f"- {chunk['scale']}: {chunk['text']}...")   

In [ ]:
# genrate query

# 4-bit квантизация для экономии памяти
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

def generate_questions(chunk_text, num_questions=3):
    prompt = f"""Ты — помощник, который генерирует вопросы к рецептам. 
На основе текста рецепта придумай {num_questions} вопроса, на которые отвечает этот текст.

Текст рецепта: {chunk_text}

Вопросы (каждый с новой строки, без нумерации):"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.8,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    questions = response.replace(prompt, '').strip().split('\n')
    return [q.strip() for q in questions if q.strip() and '?' in q][:num_questions]

In [ ]:
for i in range(3):
  print(generate_questions(chunks[i]))

In [ ]:
# use model to create lable for query

In [ ]:
class PovaryoshkaRetriever(torch.nn.Module):
  def __init__(self):
    ...
  
  def forward(self):
    ...
